# Breakeven Beta Scan

Beta-weighted analysis of breakeven inflation (5y, 10y, 30y) vs duration and crude oil.

**Goal:** Net out duration and oil moves from breakevens to isolate the "pure" inflation residual.

- Individual factor scans (each BE vs each factor)
- Sequential netting: regress out matched-tenor duration first, then crude → pure BE residual

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "stats").exists():
    for p in ROOT.parents:
        if (p / "stats").exists():
            ROOT = p
            break
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from stats import roll_lr_diff, ou_zscore, roll_half_life, ou_params, ou_summary
from stats import fit_pca, explain, residual_from_pca
from utils.helpers import query_db
from utils.viz import Viz

START = "2001-01-01"
LOOKBACKS = [60, 120, 252, 504]
PLOT_LOOKBACK = 252
TO_BPS = True

MIN_R2_CORR = 0.02
MAX_BETA_CV = 1.50
BETA_CV_CAP = 2.00
ACTION_Z = 1.50

# Breakeven tickers and their matched-tenor duration hedge
BE_UNIVERSE = {
    "5y_be":  {"be": "USGGBE05Y Index", "dur": "USGG5YR Index"},
    "10y_be": {"be": "USGGBE10Y Index", "dur": "USGG10YR Index"},
    "30y_be": {"be": "USGGBE30Y Index", "dur": "USGG30YR Index"},
}

CRUDE = "CL1 Comdty"

In [ ]:
def pull_px(tickers: list[str], start: str) -> pd.DataFrame:
    q = """
    SELECT ts, ticker, px_last
    FROM md.index_eod
    WHERE ticker = ANY(%s)
      AND ts >= %s
    ORDER BY ts
    """
    df = query_db(q, params=(tickers, start))
    if df.empty:
        return pd.DataFrame()
    df["ts"] = pd.to_datetime(df["ts"])
    return df.pivot(index="ts", columns="ticker", values="px_last").sort_index()


def fit_pair(px: pd.DataFrame, y_col: str, x_col: str, lookback: int) -> pd.DataFrame | None:
    pair = px[[y_col, x_col]].dropna().copy()
    if len(pair) < lookback + 10:
        return None

    raw = roll_lr_diff(pair[x_col], pair[y_col], lookback=lookback).to_pandas()
    raw.index = pair.index[1:]  # first row lost to diff

    reg = pd.DataFrame(index=raw.index)
    reg["y"] = raw["y"]
    reg["x"] = raw["x"]
    reg["alpha"] = raw["alpha"]
    reg["beta"] = raw["beta"]
    reg["resid"] = raw["resid_cum"]

    z = ou_zscore(reg["resid"], lookback=lookback).to_pandas()
    z.index = reg.index
    reg["ou_z"] = z

    reg["r2_corr"] = raw["r2"].clip(lower=0.0, upper=1.0)

    beta_abs_mean = reg["beta"].abs().rolling(lookback).mean()
    reg["beta_cv"] = (reg["beta"].rolling(lookback).std() / beta_abs_mean).replace(
        [np.inf, -np.inf], np.nan
    )

    reg["fair"] = reg["y"] - reg["resid"]
    return reg


def fit_sequential(px: pd.DataFrame, y_col: str, x_cols: list[str], lookback: int) -> pd.DataFrame | None:
    """Sequentially regress out each factor from y. Returns final cumulative residual + z-score."""
    resid = px[y_col].copy()
    all_betas = {}

    for x_col in x_cols:
        pair = pd.concat([resid, px[x_col]], axis=1).dropna()
        pair.columns = ["y", "x"]
        if len(pair) < lookback + 10:
            return None

        raw = roll_lr_diff(pair["x"], pair["y"], lookback=lookback).to_pandas()
        raw.index = pair.index[1:]

        # update residual to cumulative residual from this step
        resid = raw["resid_cum"].copy()
        resid.index = raw.index
        all_betas[x_col] = raw["beta"].copy()
        all_betas[x_col].index = raw.index

    reg = pd.DataFrame(index=resid.index)
    reg["y"] = px[y_col].reindex(reg.index)
    reg["resid"] = resid

    z = ou_zscore(reg["resid"], lookback=lookback).to_pandas()
    z.index = reg.index
    reg["ou_z"] = z

    for x_col, b in all_betas.items():
        reg[f"beta_{x_col}"] = b

    reg["fair"] = reg["y"] - reg["resid"]
    return reg

In [ ]:
# Pull all data in one shot
all_tickers = (
    [v["be"] for v in BE_UNIVERSE.values()]
    + [v["dur"] for v in BE_UNIVERSE.values()]
    + [CRUDE]
)
all_tickers = list(dict.fromkeys(all_tickers))  # dedupe

px = pull_px(all_tickers, START)
if TO_BPS:
    # scale yields/BEs to bps, but NOT crude
    yield_cols = [c for c in px.columns if c != CRUDE]
    px[yield_cols] = px[yield_cols] * 100.0

last_date = px.index.max().date()
print(f"Data through: {last_date}")
print(f"Tickers: {list(px.columns)}")
if pd.Timestamp(last_date) < pd.Timestamp.today().normalize() - pd.Timedelta(days=3):
    print("WARNING: data looks stale vs today.")

## Individual Factor Scans

Each breakeven regressed against each factor independently across all lookback windows.

In [ ]:
# Run individual factor regressions for each BE tenor
results: dict[tuple[str, str, int], pd.DataFrame] = {}  # (be_name, factor_label, lookback)
rows = []

for be_name, spec in BE_UNIVERSE.items():
    be_ticker = spec["be"]
    dur_ticker = spec["dur"]
    if be_ticker not in px.columns:
        print(f"[skip] {be_name}: {be_ticker} not in data")
        continue

    factors = {"dur": dur_ticker, "crude": CRUDE}

    for factor_label, factor_ticker in factors.items():
        if factor_ticker not in px.columns:
            print(f"[skip] {be_name} ~ {factor_label}: {factor_ticker} not in data")
            continue

        for lb in LOOKBACKS:
            reg = fit_pair(px, be_ticker, factor_ticker, lb)
            if reg is None:
                continue
            results[(be_name, factor_label, lb)] = reg

            tail = reg.dropna(subset=["beta", "r2_corr", "beta_cv", "resid", "ou_z"])
            if tail.empty:
                continue
            last = tail.iloc[-1]
            active = bool(
                (float(last["r2_corr"]) >= MIN_R2_CORR)
                and (float(last["beta_cv"]) <= MAX_BETA_CV)
            )
            rows.append({
                "be": be_name,
                "factor": factor_label,
                "lookback": lb,
                "beta": float(last["beta"]),
                "r2_corr": float(last["r2_corr"]),
                "beta_cv": float(last["beta_cv"]),
                "resid": float(last["resid"]),
                "ou_z": float(last["ou_z"]),
                "active": active,
                "date": tail.index[-1].date(),
            })

summary = pd.DataFrame(rows).sort_values(["be", "factor", "lookback"]).reset_index(drop=True)
display(summary)

In [ ]:
v = Viz()

def plot(*args, **kwargs):
    display(v.line(*args, **kwargs))

# Individual factor charts at PLOT_LOOKBACK — just z-scores, skip the rest
for be_name, spec in BE_UNIVERSE.items():
    be_ticker = spec["be"]
    for factor_label in ["dur", "crude"]:
        reg = results.get((be_name, factor_label, PLOT_LOOKBACK))
        if reg is None:
            continue

        plot(
            reg[["ou_z"]].dropna(),
            title=f"OU z-score: {be_name} ~ {factor_label} ({PLOT_LOOKBACK}d)",
            residual=True,
        )
        # panel = pd.DataFrame(index=reg.index)
        # panel[be_name] = reg["y"]
        # panel["fair"] = reg["fair"]
        # plot(panel[[be_name, "fair"]].dropna(), title=f"{be_name} vs fair ({factor_label}, {PLOT_LOOKBACK}d)")
        # plot(reg[["resid"]].dropna(), title=f"Residual: {be_name} ~ {factor_label} ({PLOT_LOOKBACK}d)", residual=True)
        # plot(reg[["beta"]].dropna(), title=f"Rolling beta: {be_name} vs {factor_label} ({PLOT_LOOKBACK}d)")
        # plot(reg[["r2_corr"]].dropna(), title=f"Rolling R²: {be_name} vs {factor_label} ({PLOT_LOOKBACK}d)")

## Sequential Netting: Duration + Crude

For each BE tenor:
1. Regress BE on matched-tenor duration → get duration-adjusted residual
2. Regress that residual on crude → get "pure" BE residual (netted of both)

This isolates what breakevens are doing *after* removing the mechanical duration and oil moves.

In [ ]:
# Sequential netting: duration first, then crude
netted: dict[tuple[str, int], pd.DataFrame] = {}
net_rows = []

for be_name, spec in BE_UNIVERSE.items():
    be_ticker = spec["be"]
    dur_ticker = spec["dur"]
    if be_ticker not in px.columns:
        continue

    x_cols = [dur_ticker, CRUDE]
    missing = [c for c in x_cols if c not in px.columns]
    if missing:
        print(f"[skip] {be_name}: missing {missing}")
        continue

    for lb in LOOKBACKS:
        reg = fit_sequential(px, be_ticker, x_cols, lb)
        if reg is None:
            continue
        netted[(be_name, lb)] = reg

        tail = reg.dropna(subset=["resid", "ou_z"])
        if tail.empty:
            continue
        last = tail.iloc[-1]

        row = {
            "be": be_name,
            "lookback": lb,
            "resid": float(last["resid"]),
            "ou_z": float(last["ou_z"]),
            "date": tail.index[-1].date(),
        }
        # include betas for each factor
        for x_col in x_cols:
            bcol = f"beta_{x_col}"
            if bcol in last.index and pd.notna(last[bcol]):
                label = "dur" if x_col == dur_ticker else "crude"
                row[f"beta_{label}"] = float(last[bcol])
        net_rows.append(row)

net_summary = pd.DataFrame(net_rows).sort_values(["be", "lookback"]).reset_index(drop=True)
print("Sequential netting: BE ~ duration, then crude")
display(net_summary)

In [ ]:
# Netted charts — just fair value and z-score per tenor
for be_name in BE_UNIVERSE.keys():
    reg = netted.get((be_name, PLOT_LOOKBACK))
    if reg is None:
        continue

    panel = pd.DataFrame(index=reg.index)
    panel[be_name] = reg["y"]
    panel["fair (netted)"] = reg["fair"]

    plot(
        panel[[be_name, "fair (netted)"]].dropna(),
        title=f"{be_name} vs netted fair (dur+crude, {PLOT_LOOKBACK}d)",
    )
    plot(
        reg[["ou_z"]].dropna(),
        title=f"Netted OU z-score: {be_name} (dur+crude, {PLOT_LOOKBACK}d)",
        residual=True,
    )
    # plot(reg[["resid"]].dropna(), title=f"Netted residual: {be_name} (dur+crude, {PLOT_LOOKBACK}d)", residual=True)
    # beta_cols = [c for c in reg.columns if c.startswith("beta_")]
    # if beta_cols:
    #     plot(reg[beta_cols].dropna(), title=f"Rolling betas: {be_name} ({PLOT_LOOKBACK}d)")

## Cross-Tenor Comparison

Overlay the netted z-scores for all three BE tenors to compare richness/cheapness across the curve.

In [ ]:
# Cross-tenor overlay — just the z-scores
z_overlay = pd.DataFrame()
for be_name in BE_UNIVERSE.keys():
    reg = netted.get((be_name, PLOT_LOOKBACK))
    if reg is not None:
        z_overlay[be_name] = reg["ou_z"]

if not z_overlay.empty:
    plot(
        z_overlay.dropna(how="all"),
        title=f"Netted OU z-scores: all BE tenors ({PLOT_LOOKBACK}d)",
        residual=True,
    )

# Latest snapshot
print(f"\nLatest netted z-scores ({PLOT_LOOKBACK}d):")
for be_name in BE_UNIVERSE.keys():
    reg = netted.get((be_name, PLOT_LOOKBACK))
    if reg is None:
        continue
    tail = reg.dropna(subset=["ou_z"])
    if tail.empty:
        continue
    last = tail.iloc[-1]
    z = float(last["ou_z"])
    if z >= ACTION_Z:
        action = "RICH → short BE"
    elif z <= -ACTION_Z:
        action = "CHEAP → long BE"
    else:
        action = "flat"
    print(f"  {be_name:8s}  z={z:+.3f}  resid={float(last['resid']):+.1f}bps  {action}")

## PCA on Netted BE Residuals

Run PCA on the 3 netted residual series (5y, 10y, 30y BE after removing duration + crude).

- **PC1** = level (all BEs rich/cheap together) — hard to trade without full hedges
- **PC2** = slope (front vs back of BE curve) — tradeable as BE spreads
- **PC3** = curvature — tradeable as BE butterfly

OU z-score on each PC gives half-life and expected reversion for trade timing.

In [ ]:
# Build matrix of netted residuals for PCA
resid_matrix = pd.DataFrame()
for be_name in BE_UNIVERSE.keys():
    reg = netted.get((be_name, PLOT_LOOKBACK))
    if reg is not None:
        resid_matrix[be_name] = reg["resid"]

resid_matrix = resid_matrix.dropna()
print(f"Residual matrix: {resid_matrix.shape[0]} obs x {resid_matrix.shape[1]} series")
print(f"Date range: {resid_matrix.index.min().date()} → {resid_matrix.index.max().date()}")

# Fit PCA on changes (diffs) of the netted residuals
pca_result = fit_pca(resid_matrix, n_components=3, use_changes=True)

print("\nVariance explained:")
display(explain(pca_result).to_pandas())

print("\nLoadings:")
display(pca_result["loadings"].to_pandas())

In [ ]:
# Project raw changes onto PCA loadings (without centering — so cumsum is meaningful)
loadings_np = pca_result["loadings"].drop("column").to_numpy()  # (3 tenors, 3 PCs)

# Raw daily changes of netted residuals
resid_changes = resid_matrix.diff().iloc[1:]  # drop first NaN row

# Project: each day's change vector × loadings = PC score for that day
scores_raw = pd.DataFrame(
    resid_changes.values @ loadings_np,
    index=resid_changes.index,
    columns=[f"PC{i+1}" for i in range(loadings_np.shape[1])],
)

# Cumulative sum = "level" of each PC over time
scores_cum = scores_raw.cumsum()
pc_names = list(scores_cum.columns)

print("Last 5 rows of cumulative PC scores:")
display(scores_cum.tail())

# OU analysis on each PC
print("\nOU parameters for each PC:")
for pc in pc_names:
    s = scores_cum[pc].dropna()
    ou = ou_params(s)
    print(
        f"  {pc}: theta={ou['theta']:.4f}  mu={ou['mu']:.2f}  "
        f"sigma={ou['sigma']:.4f}  half_life={ou['half_life']:.1f}d"
    )

# OU z-scores for each PC
pc_z = pd.DataFrame(index=scores_cum.index)
pc_hl = pd.DataFrame(index=scores_cum.index)

for pc in pc_names:
    z = ou_zscore(scores_cum[pc], lookback=PLOT_LOOKBACK).to_pandas()
    z.index = scores_cum.index
    pc_z[f"{pc}_z"] = z

    hl = roll_half_life(scores_cum[pc], lookback=PLOT_LOOKBACK).to_pandas()
    hl.index = scores_cum.index
    pc_hl[f"{pc}_hl"] = hl

# Latest snapshot
print(f"\nLatest PC z-scores ({PLOT_LOOKBACK}d):")
for pc in pc_names:
    z_col = f"{pc}_z"
    hl_col = f"{pc}_hl"
    latest_z = pc_z[z_col].dropna()
    latest_hl = pc_hl[hl_col].dropna()
    if latest_z.empty:
        continue
    z_now = float(latest_z.iloc[-1])
    hl_now = float(latest_hl.iloc[-1]) if not latest_hl.empty else np.nan

    if abs(z_now) >= ACTION_Z:
        direction = "SHORT" if z_now > 0 else "LONG"
        action = f"{direction} {pc}"
    else:
        action = "flat"

    print(
        f"  {pc}: z={z_now:+.3f}  half_life={hl_now:.1f}d  {action}"
    )

In [ ]:
# PCA charts — just the z-scores overlay and PC2/PC3 z-scores
plot(
    pc_z.dropna(how="all"),
    title=f"PC OU z-scores ({PLOT_LOOKBACK}d)",
    residual=True,
)

# PC2 and PC3 are the tradeable ones — just z-scores
for pc in ["PC1", "PC2", "PC3"]:
    z_col = f"{pc}_z"
    hl_col = f"{pc}_hl"
    if z_col not in pc_z.columns:
        continue

    plot(
        pc_z[[z_col]].dropna(),
        title=f"{pc} OU z-score ({PLOT_LOOKBACK}d)",
        residual=True,
    )
    # plot(scores_cum[[pc]].dropna(), title=f"{pc} score (cumulative)", residual=True)
    # plot(pc_hl[[hl_col]].dropna().clip(lower=0, upper=200), title=f"{pc} rolling half-life ({PLOT_LOOKBACK}d)")

In [ ]:
# Trade construction: map PC signals back to BE tenor weights
be_names = list(BE_UNIVERSE.keys())

print("Trade construction guide:")
print("=" * 60)

for pc in pc_names:
    z_col = f"{pc}_z"
    hl_col = f"{pc}_hl"
    latest_z = pc_z[z_col].dropna()
    latest_hl = pc_hl[hl_col].dropna()
    if latest_z.empty:
        continue

    z_now = float(latest_z.iloc[-1])
    hl_now = float(latest_hl.iloc[-1]) if not latest_hl.empty else np.nan

    if abs(z_now) < 0.5:
        continue  # skip if no signal

    # PC weights for each tenor (from loadings_np computed above)
    pc_idx = int(pc[-1]) - 1
    weights = loadings_np[:, pc_idx]
    # If z > 0, PC is rich → short the PC → flip signs
    # If z < 0, PC is cheap → long the PC → keep signs
    trade_sign = -1.0 if z_now > 0 else 1.0
    trade_weights = weights * trade_sign
    # Normalize so largest weight = 1
    max_w = np.abs(trade_weights).max()
    if max_w > 0:
        trade_weights = trade_weights / max_w

    print(f"\n{pc}: z={z_now:+.3f}  half_life={hl_now:.1f}d")
    print(f"  Expected reversion: ~{abs(z_now)*0.5:.2f}σ in {hl_now:.0f}d")
    for i, be_name in enumerate(be_names):
        w = trade_weights[i]
        direction = "LONG" if w > 0 else "SHORT"
        print(f"    {be_name:8s}: {direction} {abs(w):.3f}")
    print()

## Beta Mean-Reversion Analysis

The rolling beta itself tends to mean-revert. If a beta is stretched (e.g., 10y BE overreacting to duration), 
the leg with the inflated beta has more convexity when beta normalizes.

- OU z-score on the beta series → flag when beta is dislocated
- Short half-life = beta will snap back → overweight that leg for convexity
- Long half-life or trending = possible regime shift → don't fight it

In [ ]:
# OU analysis on rolling betas themselves
beta_ou_rows = []

for be_name, spec in BE_UNIVERSE.items():
    dur_ticker = spec["dur"]
    factors = {"dur": dur_ticker, "crude": CRUDE}

    for factor_label, factor_ticker in factors.items():
        reg = results.get((be_name, factor_label, PLOT_LOOKBACK))
        if reg is None:
            continue

        beta_series = reg["beta"].dropna()
        if len(beta_series) < PLOT_LOOKBACK:
            continue

        # OU params on the beta
        ou = ou_params(beta_series)

        # OU z-score on beta
        beta_z = ou_zscore(beta_series, lookback=PLOT_LOOKBACK).to_pandas()
        beta_z.index = beta_series.index

        # Rolling half-life of the beta
        beta_hl = roll_half_life(beta_series, lookback=PLOT_LOOKBACK).to_pandas()
        beta_hl.index = beta_series.index

        latest_beta = float(beta_series.iloc[-1])
        latest_z = float(beta_z.dropna().iloc[-1]) if not beta_z.dropna().empty else np.nan
        latest_hl = float(beta_hl.dropna().iloc[-1]) if not beta_hl.dropna().empty else np.nan

        beta_ou_rows.append({
            "be": be_name,
            "factor": factor_label,
            "beta_now": latest_beta,
            "beta_z": latest_z,
            "beta_hl": latest_hl,
            "ou_theta": ou["theta"],
            "ou_mu": ou["mu"],
            "ou_hl": ou["half_life"],
        })

beta_ou_df = pd.DataFrame(beta_ou_rows).sort_values(["be", "factor"]).reset_index(drop=True)
print("Beta mean-reversion analysis:")
display(beta_ou_df)

# Flag stretched betas
BETA_Z_THRESH = 1.0
stretched = beta_ou_df[beta_ou_df["beta_z"].abs() >= BETA_Z_THRESH].copy()
if not stretched.empty:
    print(f"\nStretched betas (|z| >= {BETA_Z_THRESH}):")
    for _, row in stretched.iterrows():
        direction = "HIGH" if row["beta_z"] > 0 else "LOW"
        convexity_leg = row["be"]
        print(
            f"  {row['be']} ~ {row['factor']}: beta={row['beta_now']:.4f} "
            f"z={row['beta_z']:+.3f} ({direction}) "
            f"hl={row['beta_hl']:.0f}d → {convexity_leg} has convexity if beta reverts"
        )
else:
    print("\nNo stretched betas currently.")

In [ ]:
# Beta z-scores overlay — one chart per factor, skip individual half-lives
for factor_label in ["dur", "crude"]:
    beta_z_overlay = pd.DataFrame()

    for be_name in BE_UNIVERSE.keys():
        reg = results.get((be_name, factor_label, PLOT_LOOKBACK))
        if reg is None:
            continue

        beta_series = reg["beta"].dropna()
        if len(beta_series) < PLOT_LOOKBACK:
            continue

        bz = ou_zscore(beta_series, lookback=PLOT_LOOKBACK).to_pandas()
        bz.index = beta_series.index
        beta_z_overlay[be_name] = bz

    if not beta_z_overlay.empty:
        plot(
            beta_z_overlay.dropna(how="all"),
            title=f"Beta OU z-scores vs {factor_label} ({PLOT_LOOKBACK}d)",
            residual=True,
        )
    # beta_overlay / beta_hl_overlay charts commented out
    # plot(beta_overlay.dropna(how="all"), title=f"Rolling betas vs {factor_label} ({PLOT_LOOKBACK}d)")
    # plot(beta_hl_overlay.dropna(how="all").clip(lower=0, upper=200), title=f"Beta half-lives vs {factor_label} ({PLOT_LOOKBACK}d)")

## Combined Signal: Residual + Beta Dislocation

When both signals align:
- **Residual z-score** says a BE tenor is cheap/rich vs factors
- **Beta z-score** says the relationship is stretched → convexity on reversion

This is the highest conviction setup: trade the spread AND overweight the leg with the stretched beta.

In [ ]:
# Combined signal dashboard
print("COMBINED SIGNAL DASHBOARD")
print("=" * 80)
print(f"{'BE':8s} {'factor':8s} {'resid_z':>8s} {'beta_z':>8s} {'beta_hl':>8s} {'signal':>20s}")
print("-" * 80)

for be_name, spec in BE_UNIVERSE.items():
    dur_ticker = spec["dur"]
    factors = {"dur": dur_ticker, "crude": CRUDE}

    for factor_label, factor_ticker in factors.items():
        reg = results.get((be_name, factor_label, PLOT_LOOKBACK))
        if reg is None:
            continue

        # Residual z
        resid_tail = reg[["ou_z"]].dropna()
        if resid_tail.empty:
            continue
        resid_z = float(resid_tail.iloc[-1, 0])

        # Beta z and half-life
        beta_series = reg["beta"].dropna()
        if len(beta_series) < PLOT_LOOKBACK:
            continue

        bz = ou_zscore(beta_series, lookback=PLOT_LOOKBACK).to_pandas()
        bz.index = beta_series.index
        beta_z = float(bz.dropna().iloc[-1])

        bhl = roll_half_life(beta_series, lookback=PLOT_LOOKBACK).to_pandas()
        bhl.index = beta_series.index
        beta_hl = float(bhl.dropna().iloc[-1]) if not bhl.dropna().empty else np.nan

        # Combined signal logic
        signal = ""
        resid_strong = abs(resid_z) >= ACTION_Z
        beta_stretched = abs(beta_z) >= BETA_Z_THRESH
        beta_fast = beta_hl < 60  # quick reversion expected

        if resid_strong and beta_stretched and beta_fast:
            resid_dir = "CHEAP" if resid_z < 0 else "RICH"
            beta_dir = "HIGH" if beta_z > 0 else "LOW"
            signal = f"*** {resid_dir} + beta {beta_dir} ***"
        elif resid_strong:
            resid_dir = "CHEAP" if resid_z < 0 else "RICH"
            signal = f"{resid_dir} (resid only)"
        elif beta_stretched and beta_fast:
            beta_dir = "HIGH" if beta_z > 0 else "LOW"
            signal = f"beta {beta_dir} (convexity)"
        else:
            signal = "—"

        print(
            f"{be_name:8s} {factor_label:8s} "
            f"{resid_z:+8.3f} {beta_z:+8.3f} {beta_hl:8.1f} "
            f"{signal:>20s}"
        )

## Residual Distributions

Distribution of netted residuals — are they normal? Skewed? Fat-tailed?

In [ ]:
# Netted residual distributions — all tenors side by side
resid_dist_df = pd.DataFrame()
for be_name in BE_UNIVERSE.keys():
    reg = netted.get((be_name, PLOT_LOOKBACK))
    if reg is not None:
        resid_dist_df[be_name] = reg["resid"]

v.residual_dist(resid_dist_df.dropna(), title="Netted residual distributions (dur+crude)")

# OU z-score distributions
z_dist_df = pd.DataFrame()
for be_name in BE_UNIVERSE.keys():
    reg = netted.get((be_name, PLOT_LOOKBACK))
    if reg is not None:
        z_dist_df[be_name] = reg["ou_z"]

v.residual_dist(z_dist_df.dropna(), title="Netted OU z-score distributions")

# PC score distributions
v.residual_dist(scores_cum.dropna(), title="PC score distributions (cumulative)")

## Signal Stack: Full Book View

Three signal layers combined into a single trade blotter:

1. **PC signals** (curve RV) — duration-hedged BE spread/fly trades from PC2/PC3 loadings
2. **Oil-neutral signal** (outright enrichment) — netted residual per tenor, used as conviction modifier
3. **Beta mean-reversion** (convexity overlay) — stretched betas flag which leg to overweight

In [ ]:
# ---------------------------------------------------------------------------
# Gather all signal components into one place
# ---------------------------------------------------------------------------

be_names = list(BE_UNIVERSE.keys())

# 1. Oil-neutral (netted) z-scores per tenor
oil_neutral = {}
for be_name in be_names:
    reg = netted.get((be_name, PLOT_LOOKBACK))
    if reg is None:
        continue
    tail = reg.dropna(subset=["ou_z"])
    if tail.empty:
        continue
    last = tail.iloc[-1]
    hl = roll_half_life(reg["resid"].dropna(), lookback=PLOT_LOOKBACK).to_pandas()
    hl.index = reg["resid"].dropna().index
    oil_neutral[be_name] = {
        "z": float(last["ou_z"]),
        "resid": float(last["resid"]),
        "hl": float(hl.dropna().iloc[-1]) if not hl.dropna().empty else np.nan,
    }

# 2. Duration-only z-scores per tenor
dur_only = {}
for be_name, spec in BE_UNIVERSE.items():
    reg = results.get((be_name, "dur", PLOT_LOOKBACK))
    if reg is None:
        continue
    tail = reg.dropna(subset=["ou_z"])
    if tail.empty:
        continue
    last = tail.iloc[-1]
    dur_only[be_name] = {
        "z": float(last["ou_z"]),
        "beta": float(last["beta"]),
        "resid": float(last["resid"]),
    }

# 3. Beta dislocation per tenor per factor
beta_dislocations = {}
for be_name, spec in BE_UNIVERSE.items():
    for factor_label in ["dur", "crude"]:
        reg = results.get((be_name, factor_label, PLOT_LOOKBACK))
        if reg is None:
            continue
        beta_s = reg["beta"].dropna()
        if len(beta_s) < PLOT_LOOKBACK:
            continue
        bz = ou_zscore(beta_s, lookback=PLOT_LOOKBACK).to_pandas()
        bz.index = beta_s.index
        bhl = roll_half_life(beta_s, lookback=PLOT_LOOKBACK).to_pandas()
        bhl.index = beta_s.index
        beta_dislocations[(be_name, factor_label)] = {
            "beta": float(beta_s.iloc[-1]),
            "z": float(bz.dropna().iloc[-1]),
            "hl": float(bhl.dropna().iloc[-1]) if not bhl.dropna().empty else np.nan,
        }

# 4. PC signals
pc_signals = {}
for pc in pc_names:
    z_col = f"{pc}_z"
    hl_col = f"{pc}_hl"
    z_s = pc_z[z_col].dropna()
    hl_s = pc_hl[hl_col].dropna()
    if z_s.empty:
        continue
    pc_idx = int(pc[-1]) - 1
    pc_signals[pc] = {
        "z": float(z_s.iloc[-1]),
        "hl": float(hl_s.iloc[-1]) if not hl_s.empty else np.nan,
        "loadings": loadings_np[:, pc_idx],
        "var_pct": float(pca_result["explained_variance"][pc_idx] * 100),
    }

In [ ]:
# ---------------------------------------------------------------------------
# BOOK BLOTTER — all signal layers combined
# ---------------------------------------------------------------------------

print(f"BE INFLATION SIGNAL BOOK — {last_date}")
print("=" * 90)

# --- Layer 1: PC curve RV trades ---
print("\n┌─ LAYER 1: PC CURVE RV (duration-hedged BE spread/fly)")
print("│")
for pc, sig in pc_signals.items():
    z, hl = sig["z"], sig["hl"]
    if abs(z) < 0.75:
        continue

    trade_sign = -1.0 if z > 0 else 1.0
    weights = sig["loadings"] * trade_sign
    max_w = np.abs(weights).max()
    if max_w > 0:
        weights = weights / max_w

    status = "ACTIVE" if abs(z) >= ACTION_Z else "WATCH"
    print(f"│  {pc} ({sig['var_pct']:.1f}% var): z={z:+.3f}  hl={hl:.0f}d  [{status}]")

    for i, be_name in enumerate(be_names):
        w = weights[i]
        d = "L" if w > 0 else "S"

        # Check if oil-neutral signal confirms this direction
        on = oil_neutral.get(be_name, {})
        on_z = on.get("z", 0)
        confirms_dir = (w > 0 and on_z < -1.0) or (w < 0 and on_z > 1.0)
        confirm_tag = f"  ← oil-neutral confirms (z={on_z:+.2f})" if confirms_dir else ""

        # Check if beta is stretched on this leg
        bd_dur = beta_dislocations.get((be_name, "dur"), {})
        bd_crude = beta_dislocations.get((be_name, "crude"), {})
        convex_tags = []
        for label, bd in [("dur", bd_dur), ("crude", bd_crude)]:
            if bd and abs(bd.get("z", 0)) >= BETA_Z_THRESH and bd.get("hl", 999) < 60:
                convex_tags.append(f"β_{label} z={bd['z']:+.2f} hl={bd['hl']:.0f}d")
        convex_str = f"  ← CONVEXITY: {', '.join(convex_tags)}" if convex_tags else ""

        print(f"│    {d} {abs(w):.3f} {be_name:8s}{confirm_tag}{convex_str}")
    print("│")

# --- Layer 2: Oil-neutral outright signals ---
print("├─ LAYER 2: OIL-NEUTRAL OUTRIGHT (dur+crude netted)")
print("│")
for be_name in be_names:
    on = oil_neutral.get(be_name)
    if on is None:
        continue
    z, resid, hl = on["z"], on["resid"], on["hl"]
    if abs(z) < 1.0:
        tag = "flat"
    elif abs(z) >= ACTION_Z:
        tag = "ACTIVE"
    else:
        tag = "WATCH"

    direction = "CHEAP" if z < 0 else "RICH"
    print(f"│  {be_name:8s}: z={z:+.3f}  resid={resid:+.1f}bps  hl={hl:.0f}d  {direction} [{tag}]")
print("│")

# --- Layer 3: Beta convexity overlay ---
print("├─ LAYER 3: BETA CONVEXITY (stretched betas with fast reversion)")
print("│")
has_any = False
for be_name in be_names:
    for factor_label in ["dur", "crude"]:
        bd = beta_dislocations.get((be_name, factor_label))
        if bd is None:
            continue
        bz, bhl, beta = bd["z"], bd["hl"], bd["beta"]
        if abs(bz) < BETA_Z_THRESH:
            continue
        has_any = True
        speed = "FAST" if bhl < 60 else "SLOW"
        direction = "HIGH" if bz > 0 else "LOW"
        print(
            f"│  {be_name:8s} ~ {factor_label:6s}: β={beta:.4f}  "
            f"β_z={bz:+.3f}  β_hl={bhl:.0f}d  {direction} [{speed}]"
        )
if not has_any:
    print("│  (no stretched betas)")
print("│")

# --- Summary ---
print("└─ SUMMARY")
print()

# Count active signals
active_pc = sum(1 for s in pc_signals.values() if abs(s["z"]) >= ACTION_Z)
active_outright = sum(1 for s in oil_neutral.values() if abs(s["z"]) >= ACTION_Z)
active_beta = sum(
    1 for bd in beta_dislocations.values()
    if abs(bd["z"]) >= BETA_Z_THRESH and bd["hl"] < 60
)

print(f"   Active PC signals:       {active_pc}/{len(pc_signals)}")
print(f"   Active outright signals: {active_outright}/{len(oil_neutral)}")
print(f"   Fast beta dislocations:  {active_beta}/{len(beta_dislocations)}")